# 05b - Base RAG Error Analysis

Re-evaluates Base RAG predictions with improved citation heuristics and creates an automatic error analysis file for the report.

In [ ]:
from pathlib import Path
import json
import sys
import importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path = [str(DRIVE_ROOT)] + [p for p in sys.path if p != str(DRIVE_ROOT)]
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()

output_dir = DRIVE_ROOT / 'outputs/generation_eval'
predictions_csv = output_dir / 'base_rag_predictions_v1.csv'
if not predictions_csv.exists():
    raise FileNotFoundError(predictions_csv)

predictions_csv

In [ ]:
from src.evaluation_qa import evaluate_generation_predictions
from src.error_analysis import make_error_analysis

summary = evaluate_generation_predictions(
    predictions_csv=predictions_csv,
    output_eval_csv=output_dir / 'base_rag_eval_v1.csv',
    output_summary_json=output_dir / 'base_rag_eval_summary_v1.json',
)

errors = make_error_analysis(
    eval_csv=output_dir / 'base_rag_eval_v1.csv',
    output_csv=DRIVE_ROOT / 'reports/error_analysis_base_rag.csv',
    per_type_limit=30,
)

summary

In [ ]:
import pandas as pd

eval_df = pd.read_csv(output_dir / 'base_rag_eval_v1.csv', dtype=str, keep_default_na=False)
print(eval_df['error_type_auto'].value_counts().to_string())
errors[['question_id', 'topic', 'error_type_auto', 'gold_article_keys', 'retrieved_citations']].head(20)

Expected outputs:

- `outputs/generation_eval/base_rag_eval_v1.csv`
- `outputs/generation_eval/base_rag_eval_summary_v1.json`
- `reports/error_analysis_base_rag.csv`